# Phase 2 — Data Preprocessing
**Goal:** Clean data, scale features, apply SMOTE to fix imbalance.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib, warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

df = pd.read_csv('../data/creditcard.csv')
print(f"Dataset loaded: {df.shape}")

In [ ]:
# Step 1 — Check nulls
print("=== Null check ===")
print(df.isnull().sum().sum(), "total nulls")
print("Data is clean!" if df.isnull().sum().sum()==0 else "Nulls found — handling...")
df = df.fillna(df.median(numeric_only=True))

In [ ]:
# Step 2 — Drop Time, scale Amount
df_clean = df.copy()
df_clean = df_clean.drop('Time', axis=1)

scaler = StandardScaler()
df_clean['Amount'] = scaler.fit_transform(df_clean[['Amount']])

print("Time column dropped.")
print("Amount scaled — mean:", round(df_clean['Amount'].mean(),4),
      " std:", round(df_clean['Amount'].std(),4))

In [ ]:
# Step 3 — Train test split
X = df_clean.drop('Class', axis=1)
y = df_clean['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train size : {X_train.shape[0]}")
print(f"Test size  : {X_test.shape[0]}")
print(f"Fraud in test: {y_test.sum()}")

In [ ]:
# Step 4 — SMOTE on training data ONLY
print("Before SMOTE:")
print(f"  Normal: {(y_train==0).sum()}")
print(f"  Fraud : {(y_train==1).sum()}")

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print(f"  Normal: {(y_train_bal==0).sum()}")
print(f"  Fraud : {(y_train_bal==1).sum()}")
print("Perfectly balanced!")

In [ ]:
# Visualize before vs after SMOTE
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
pd.Series(y_train).value_counts().plot(
    kind='bar', ax=axes[0], color=['steelblue','crimson'], edgecolor='black')
axes[0].set_title('Before SMOTE'); axes[0].set_xticklabels(['Normal','Fraud'],rotation=0)

pd.Series(y_train_bal).value_counts().plot(
    kind='bar', ax=axes[1], color=['steelblue','crimson'], edgecolor='black')
axes[1].set_title('After SMOTE'); axes[1].set_xticklabels(['Normal','Fraud'],rotation=0)

plt.tight_layout()
plt.savefig('../reports/phase2_smote_comparison.png', dpi=150)
plt.show()
print("SMOTE comparison saved!")

In [ ]:
# Save all outputs
joblib.dump(X_train_bal, '../data/X_train.pkl')
joblib.dump(X_test,      '../data/X_test.pkl')
joblib.dump(y_train_bal, '../data/y_train.pkl')
joblib.dump(y_test,      '../data/y_test.pkl')
joblib.dump(scaler,      '../data/scaler.pkl')
print("All data saved!")
print("Phase 2 complete!")